[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/09_functions.ipynb)

*Python from Scratch · Notebook 9 of 12*

# Functions & Recursion

Package code into reusable functions: parameters, return values, scope, lambdas, closures and recursion.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A function is a recipe: ingredients go in, a dish comes out, and the kitchen stays tidy.

**What interviewers really test here:** Default-argument traps, *args/**kwargs, closures, recursion and the call stack.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why functions? *(Basics)*
- [ ] 2. Parameters, arguments and return *(Basics)*
- [ ] 3. Flexible arguments: \*args and \*\*kwargs *(Core)*
- [ ] 4. Scope: where variables live *(Core)*
- [ ] 5. Functions are values: lambda, map, closures *(Core)*
- [ ] 6. Recursion *(Advanced)*
- [ ] 7. Making recursion fast: memoisation *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **function** is a named, reusable block of code that performs a specific task. It may take inputs (**parameters**) and may produce an output (**return value**).




- **Built-in functions**: provided by Python — `print, len, sum, max, sorted, range, input` …
- **User-defined functions**: created with `def` (or `lambda`).
- **Methods**: functions that belong to an object — `"hi".upper()`.



Benefits: **reuse** (write once), **abstraction** (hide details behind a name), **modularity** (split a program into parts), easier **testing** and **debugging**.

### Anatomy of a function

```python
def name(param1, param2=default) -> ReturnType:   # header (signature)
    """Docstring: what the function does."""
    body                                          # indented statements
    return value                                  # optional
```




- **def** creates a function object and binds it to the name.
- The **docstring** is available as `name.__doc__` and through `help(name)`.
- **Type hints** (`x: int`, `-> str`) document types; Python does not enforce them at run time.
- A function with no `return` returns `None`.

### Parameters and arguments

| Kind | Definition | Call |
| --- | --- | --- |
| Positional | `def f(a, b)` | `f(1, 2)` |
| Keyword | same | `f(b=2, a=1)` |
| Default | `def f(a, b=10)` | `f(1)` |
| Variable positional | `def f(*args)` | `f(1, 2, 3)` → args is a tuple |
| Variable keyword | `def f(**kwargs)` | `f(x=1)` → kwargs is a dict |
| Keyword-only | `def f(a, *, key)` | `f(1, key=2)` |
| Positional-only | `def f(a, /, b)` | `f(1, 2)` or `f(1, b=2)` |



Order in a definition: positional-only, `/`, normal, `*args` (or `*`), keyword-only, `**kwargs`.



**Argument passing**: Python passes **object references** ("call by sharing"). A function can mutate a mutable argument (a list), but reassigning the parameter doesn't affect the caller's variable.



**Default values are evaluated once**, when the function is defined. A mutable default (`def f(x=[])`) is shared between calls — use `None` and create the list inside.

### Scope and namespaces

A **namespace** maps names to objects. Scope decides which namespace a name is looked up in, following **LEGB**:



| Scope | Where |
| --- | --- |
| **L**ocal | inside the current function |
| **E**nclosing | inside any outer function (for nested functions) |
| **G**lobal | top level of the module |
| **B**uilt-in | Python's built-ins (`len`, `print` …) |



Assigning to a name inside a function makes it local unless declared `global` (module level) or `nonlocal` (enclosing function). Reading an outer name needs no declaration.

### First-class functions, lambdas, closures, decorators

- **First-class**: functions can be assigned, passed as arguments, returned and stored in data structures.
- **Higher-order function**: takes or returns a function — `map`, `filter`, `sorted(key=…)`.
- **Lambda**: `lambda params: expression` — an anonymous function limited to one expression.
- **Closure**: an inner function that remembers variables from its enclosing scope after the outer function has returned.
- **Decorator**: a function that takes a function and returns a new one, applied with `@decorator` above a definition. Examples: `@lru_cache`, `@property`, `@staticmethod`.

### Recursion

**Recursion** is when a function calls itself. It needs a **base case** and a **recursive case** that progresses toward it.




- Each call gets its own **stack frame** (local variables). Depth is limited (`sys.getrecursionlimit()`, default 1000) → `RecursionError`.
- Any recursion can be rewritten as a loop with an explicit stack; Python has no tail-call optimisation.
- Running time is found with a recurrence: T(n) = T(n − 1) + O(1) → O(n); T(n) = 2T(n − 1) + O(1) → O(2ⁿ); T(n) = T(n/2) + O(1) → O(log n).
- **Memoisation** caches results of a pure recursive function so repeated sub-problems are computed once.



Recursion fits naturally when the data is recursive: nested lists, trees, folders, expressions, divide and conquer.

---
# Part 2: Lessons, step by step

## Lesson 1: Why functions?

*Level: Basics*

A **function** is a named block of code you can run whenever you need it. It lets you write something once and reuse it, give a clear name to a step, and test pieces separately.



```python
def greet(name):            # define: def + name + (parameters) + colon
    message = "Hi " + name  # the indented body
    return message          # send a value back

text = greet("Ana")         # call: run the body with name = "Ana"
print(text)
```



Defining a function doesn't run it. The body runs only when you **call** it. In the animation, watch the **call stack** appear when `greet` starts and disappear when it returns.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def why_demo():
    def greet(name):
        message = "Hi " + name
        return message

    first = greet("Ana")
    second = greet("Ben")
    print(first, "|", second)


why_demo()

Hi Ana | Hi Ben


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. greet() returns 'Hi Ana'
2. greet() returns 'Hi Ben'
3. why\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: Parameters, arguments and return

*Level: Basics*

**Parameters** are the names in the definition; **arguments** are the values you pass in a call.




- **Positional**: matched by order — `area(3, 4)`.
- **Keyword**: matched by name — `area(h=4, w=3)`.
- **Default values**: `def area(w, h=1)` makes h optional.



`return` ends the function and sends a value back. A function without `return` (or with a bare `return`) gives back `None`. Return several values as a tuple: `return lo, hi`.



**⚠️ Watch out:** `print` shows a value; `return` hands it back to the caller. A function that only prints can't be used in a calculation.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def params_demo():
    def area(w, h=1):
        return w * h

    def shout(text):
        print(text.upper())

    a = area(3)
    b = area(3, 4)
    c = area(h=5, w=2)
    result = shout("hello")
    print(a, b, c, result)


params_demo()

HELLO
3 12 10 None


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. area() returns 3
2. area() returns 12
3. area() returns 10
4. shout() returns None
5. params\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 3: Flexible arguments: \*args and \*\*kwargs

*Level: Core*

```python
def total(*nums):            # nums is a tuple of all positional args
    return sum(nums)
total(1, 2, 3)

def profile(**info):         # info is a dict of all keyword args
    return info
profile(name="Ana", age=21)

def f(a, b=2, *args, key=None, **kwargs): ...   # the full order
```



The stars also work in calls: `total(*my_list)` spreads a list into arguments and `profile(**my_dict)` spreads a dict into keyword arguments.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def args_demo():
    def total(*nums):
        return sum(nums)

    def profile(**info):
        return ", ".join(f"{k}={v}" for k, v in info.items())

    values = [4, 5, 6]
    print(total(1, 2), total(*values))
    print(profile(name="Ana", city="Pune"))


args_demo()

3 15
name=Ana, city=Pune


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. total() returns 3
2. total() returns 15
3. profile() returns 'name=Ana, city=Pune'
4. args\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Scope: where variables live

*Level: Core*

Variables created inside a function are **local**: they exist only while the function runs and are invisible outside. Variables created at the top of a file are **global**.



Python looks up a name in this order — **LEGB**: **L**ocal → **E**nclosing function → **G**lobal → **B**uilt-in.



```python
count = 0
def bump():
    global count    # needed to ASSIGN to a global inside a function
    count += 1
```



Prefer passing values in and returning results over using `global`. `nonlocal` does the same for a variable of an enclosing function.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def scope_demo():
    def outer():
        x = "enclosing"

        def inner():
            y = "local"
            return x + " + " + y

        return inner()

    result = outer()
    print(result)


scope_demo()

enclosing + local


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. inner() returns 'enclosing + local'
2. outer() returns 'enclosing + local'
3. scope\_demo() returns None

</details>


## Lesson 5: Functions are values: lambda, map, closures

*Level: Core*

A function is an object like any other: you can store it in a variable, put it in a dict, or pass it to another function.



```python
square = lambda x: x * x            # a small unnamed function
sorted(words, key=len)              # pass a function as an argument
list(map(square, [1, 2, 3]))        # [1, 4, 9]
ops = {"double": lambda x: 2 * x}   # functions in a dict
```



A **closure** is an inner function that remembers variables of the function that created it:



```python
def make_multiplier(k):
    def times(x):
        return x * k       # k is remembered
    return times
triple = make_multiplier(3)
triple(5)                  # 15
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def closure_demo():
    def make_multiplier(k):
        def times(x):
            return x * k
        return times

    triple = make_multiplier(3)
    double = make_multiplier(2)
    print(triple(5), double(5), sorted(["kiwi", "fig", "banana"], key=len))


closure_demo()

15 10 ['fig', 'kiwi', 'banana']


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. make\_multiplier() returns &lt;function closure\_demo.&lt;locals&gt;.make\_...
2. make\_multiplier() returns &lt;function closure\_demo.&lt;locals&gt;.make\_...
3. times() returns 15
4. times() returns 10
5. closure\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q15](#q15), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 6: Recursion

*Level: Advanced*

A **recursive** function calls itself on a smaller problem. Every recursive function needs:




1. a **base case** that answers directly (stops the recursion), and
2. a **recursive case** that moves toward the base case.



```python
def factorial(n):
    if n <= 1:                   # base case
        return 1
    return n * factorial(n - 1)  # smaller problem
```



Watch the call stack in the animation: each call waits for the one above it, then they return one by one. Python's default limit is about 1000 nested calls (`RecursionError`).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def recursion_demo():
    def factorial(n):
        if n <= 1:
            return 1
        return n * factorial(n - 1)

    print(factorial(4))


recursion_demo()

24


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. factorial() returns 1
2. factorial() returns 2
3. factorial() returns 6
4. factorial() returns 24
5. recursion\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q6](#q6), [Q7](#q7), [Q8](#q8), [Q10](#q10), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 7: Making recursion fast: memoisation

*Level: Advanced*

Some recursions solve the same sub-problem many times. `fib(5)` calls `fib(3)` twice and `fib(2)` three times; for `fib(40)` it's hundreds of millions of calls.



**Memoisation** stores each answer the first time and reuses it:



```python
from functools import lru_cache

@lru_cache(maxsize=None)     # a decorator that adds a cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
```



This turns exponential time into linear time. It is the bridge to **dynamic programming**, covered in the DSA handbook.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def memo_demo():
    memo = {}

    def fib(n):
        if n < 2:
            return n
        if n not in memo:
            memo[n] = fib(n - 1) + fib(n - 2)
        return memo[n]

    print(fib(6), memo)


memo_demo()

8 {2: 1, 3: 2, 4: 3, 5: 5, 6: 8}


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. fib() returns 1
2. fib() returns 0
3. fib() returns 1
4. fib() returns 1
5. fib() returns 2
6. fib() returns 1
7. fib() returns 3
8. fib() returns 2
9. fib() returns 5
10. fib() returns 3
11. fib() returns 8
12. memo\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q19](#q19), [Q13](#q13), [Q18](#q18), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Greeting With a Default

🟢 Warm-up · *Default parameters*

Write `greet(name, greeting='Hello')` that returns e.g. 'Hello, Ana!'. The greeting is optional.

**Examples**

- `name = 'Ana'` → `'Hello, Ana!'`
- `name = 'Ben', greeting = 'Hi'` → `'Hi, Ben!'`

In [ ]:
def greet(name, greeting='Hello'):
    """
    Write greet(name, greeting='Hello') that returns e.g. 'Hello, Ana!'. The
    greeting is optional.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(greet, [
    (('Ana',), 'Hello, Ana!'),
    (('Ben', 'Hi'), 'Hi, Ben!'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Make greeting optional by passing None and filling it in inside.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Give the parameter a default value in the signature.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Make greeting optional by passing None and filling it in inside.

```python
def greet(name, greeting=None):
    if greeting is None:
        greeting = "Hello"
    return greeting + ", " + name + "!"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Give the parameter a default value in the signature.

```python
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Make greeting optional by passing None and filling it in inside. That costs O(1) time and O(1) space."

"I can do better: Give the parameter a default value in the signature. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/greeting-with-a-default)

<a id="q2"></a>
### Q2. Circle Area (return, don't print)

🟢 Warm-up · *return values*

Return the area of a circle of radius r, rounded to 2 decimals.

**Examples**

- `r = 1` → `3.14`
- `r = 2.5` → `19.63`
- `r = 0` → `0.0`

In [ ]:
def circle_area(r):
    """
    Return the area of a circle of radius r, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(circle_area, [
    ((1,), 3.14),
    ((2.5,), 19.63),
    ((0,), 0.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use 3.14159 typed by hand and several steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use `math.pi` and return the expression.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Use 3.14159 typed by hand and several steps.

```python
def circle_area(r):
    pi = 3.14159265
    r_squared = r * r
    area = pi * r_squared
    return round(area, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use `math.pi` and return the expression.

```python
def circle_area(r):
    import math
    return round(math.pi * r ** 2, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use 3.14159 typed by hand and several steps. That costs O(1) time and O(1) space."

"I can do better: Use `math.pi` and return the expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/circle-area-return-dont-print)

<a id="q3"></a>
### Q3. Sum Any Number of Values

🟢 Warm-up · *\*args*

Write `total(*nums)` that adds any number of arguments: total(1, 2, 3) → 6, total() → 0.

**Examples**

- `nums = 1` → `6`
- `` → `0`
- `nums = 10` → `10`

In [ ]:
def total(*nums):
    """
    Write total(*nums) that adds any number of arguments: total(1, 2, 3) → 6,
    total() → 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(total, [
    ((1, 2, 3), 6),
    ((), 0),
    ((10,), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the args tuple and add.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`*nums` collects the arguments into a tuple; pass it to `sum`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over the args tuple and add.

```python
def total(*nums):
    result = 0
    for x in nums:
        result += x
    return result
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `*nums` collects the arguments into a tuple; pass it to `sum`.

```python
def total(*nums):
    return sum(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the args tuple and add. That costs O(n) time and O(1) space."

"I can do better: `*nums` collects the arguments into a tuple; pass it to `sum`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/sum-any-number-of-values)

<a id="q4"></a>
### Q4. Factorial with Recursion

🟢 Warm-up · *Recursion · base case*

Return n! using recursion in the long way.

**Examples**

- `n = 5` → `120`
- `n = 0` → `1`
- `n = 1` → `1`

In [ ]:
def factorial(n):
    """
    Return n! using recursion in the long way.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(factorial, [
    ((5,), 120),
    ((0,), 1),
    ((1,), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: n! = n × (n − 1)!, with 0! = 1 as the base case.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.factorial` — no recursion depth limit, implemented in C.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: n! = n × (n − 1)!, with 0! = 1 as the base case.

```python
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)
```

**Cost:** O(n) time, O(n) call stack space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.factorial` — no recursion depth limit, implemented in C.

```python
def factorial(n):
    import math
    return math.factorial(n)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: n! = n × (n − 1)!, with 0! = 1 as the base case. That costs O(n) time and O(n) call stack space."

"I can do better: `math.factorial` — no recursion depth limit, implemented in C. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/factorial-with-recursion)

<a id="q5"></a>
### Q5. Sum of Digits, Recursively

🟢 Warm-up · *Recursion on numbers*

Return the sum of the digits of a non-negative integer.

**Examples**

- `n = 1234` → `10`
- `n = 0` → `0`
- `n = 9` → `9`

In [ ]:
def digit_sum(n):
    """
    Return the sum of the digits of a non-negative integer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(digit_sum, [
    ((1234,), 10),
    ((0,), 0),
    ((9,), 9),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: last digit + sum of the rest (n // 10).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Convert to a string and sum the digits.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: last digit + sum of the rest (n // 10).

```python
def digit_sum(n):
    if n < 10:
        return n
    return n % 10 + digit_sum(n // 10)
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Convert to a string and sum the digits.

```python
def digit_sum(n):
    return sum(map(int, str(n)))
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: last digit + sum of the rest (n // 10). That costs O(d) time and O(d) space."

"I can do better: Convert to a string and sum the digits. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/sum-of-digits-recursively)

<a id="q6"></a>
### Q6. Reverse a String, Recursively

🟢 Warm-up · *Recursion on strings*

Return the string reversed.

**Examples**

- `s = 'abc'` → `'cba'`
- `s = ''` → `''`
- `s = 'x'` → `'x'`

In [ ]:
def reverse(s):
    """
    Return the string reversed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse, [
    (('abc',), 'cba'),
    (('',), ''),
    (('x',), 'x'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: last character + reverse of the rest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slice with step −1.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: last character + reverse of the rest.

```python
def reverse(s):
    if s == "":
        return ""
    return s[-1] + reverse(s[:-1])
```

**Cost:** O(n²) (slices) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Slice with step −1.

```python
def reverse(s):
    return s[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: last character + reverse of the rest. That costs O(n²) (slices) time and O(n) space."

"I can do better: Slice with step −1. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/reverse-a-string-recursively)

<a id="q7"></a>
### Q7. Palindrome, Recursively

🟢 Warm-up · *Two base cases*

Return True if s reads the same backwards.

**Examples**

- `s = 'racecar'` → `True`
- `s = 'abca'` → `False`
- `s = ''` → `True`

In [ ]:
def is_pal(s):
    """
    Return True if s reads the same backwards.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_pal, [
    (('racecar',), True),
    (('abca',), False),
    (('',), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: first == last and the middle is a palindrome.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare with the reversed string.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: first == last and the middle is a palindrome.

```python
def is_pal(s):
    if len(s) <= 1:
        return True
    if s[0] != s[-1]:
        return False
    return is_pal(s[1:-1])
```

**Cost:** O(n²) (slices) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Compare with the reversed string.

```python
def is_pal(s):
    return s == s[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: first == last and the middle is a palindrome. That costs O(n²) (slices) time and O(n) space."

"I can do better: Compare with the reversed string. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/palindrome-recursively)

<a id="q8"></a>
### Q8. Countdown List, Recursively

🟢 Warm-up · *Building a list recursively*

Return [n, n − 1, …, 1] (empty for n ≤ 0).

**Examples**

- `n = 4` → `[4, 3, 2, 1]`
- `n = 0` → `[]`
- `n = 1` → `[1]`

In [ ]:
def countdown(n):
    """
    Return [n, n − 1, …, 1] (empty for n ≤ 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(countdown, [
    ((4,), [4, 3, 2, 1]),
    ((0,), []),
    ((1,), [1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: [n] + countdown(n − 1).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`list(range(n, 0, -1))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: [n] + countdown(n − 1).

```python
def countdown(n):
    if n <= 0:
        return []
    return [n] + countdown(n - 1)
```

**Cost:** O(n²) (list copies) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `list(range(n, 0, -1))`.

```python
def countdown(n):
    return list(range(n, 0, -1))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: [n] + countdown(n − 1). That costs O(n²) (list copies) time and O(n) space."

"I can do better: `list(range(n, 0, -1))`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/countdown-list-recursively)

<a id="q9"></a>
### Q9. Sort by Last Letter

🟢 Warm-up · *lambda as a key*

Sort words by their last letter (ties keep their original order).

**Examples**

- `words = ['apple', 'kiwi', 'banana', 'fig']` → `['banana', 'apple', 'fig', 'kiwi']`
- `words = []` → `[]`

In [ ]:
def by_last_letter(words):
    """
    Sort words by their last letter (ties keep their original order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(by_last_letter, [
    ((['apple', 'kiwi', 'banana', 'fig'],), ['banana', 'apple', 'fig', 'kiwi']),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Insertion sort comparing the last letters by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted(words, key=lambda w: w[-1])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Insertion sort comparing the last letters by hand.

```python
def by_last_letter(words):
    result = words[:]
    for i in range(1, len(result)):
        j = i
        while j > 0 and result[j - 1][-1] > result[j][-1]:
            result[j - 1], result[j] = result[j], result[j - 1]
            j -= 1
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sorted(words, key=lambda w: w[-1])`.

```python
def by_last_letter(words):
    return sorted(words, key=lambda w: w[-1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Insertion sort comparing the last letters by hand. That costs O(n²) time and O(n) space."

"I can do better: `sorted(words, key=lambda w: w[-1])`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/sort-by-last-letter)

<a id="q10"></a>
### Q10. GCD, Recursively

🟢 Warm-up · *Euclid's algorithm*

Return the greatest common divisor of a and b.

**Examples**

- `a = 48, b = 18` → `6`
- `a = 7, b = 3` → `1`
- `a = 10, b = 0` → `10`

In [ ]:
def gcd(a, b):
    """
    Return the greatest common divisor of a and b.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gcd, [
    ((48, 18), 6),
    ((7, 3), 1),
    ((10, 0), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive Euclid: gcd(a, b) = gcd(b, a % b), with gcd(a, 0) = a.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.gcd`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive Euclid: gcd(a, b) = gcd(b, a % b), with gcd(a, 0) = a.

```python
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)
```

**Cost:** O(log) time, O(log) stack space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.gcd`.

```python
def gcd(a, b):
    import math
    return math.gcd(a, b)
```

**Cost:** O(log) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive Euclid: gcd(a, b) = gcd(b, a % b), with gcd(a, 0) = a. That costs O(log) time and O(log) stack space."

"I can do better: `math.gcd`. That's O(log) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/gcd-recursively)

<a id="q11"></a>
### Q11. Keyword Arguments to Text

🟡 Interview standard · *\*\*kwargs*

Write `describe(**info)` that returns 'key=value' pairs sorted by key and joined by ', '.

**Examples**

- `**info = {'name': 'Ana', 'age': 21}` → `'age=21, name=Ana'`
- `**info = {}` → `''`

In [ ]:
def describe(**info):
    """
    Write describe(**info) that returns 'key=value' pairs sorted by key and
    joined by ', '.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(describe, [(({"name": "Ana", "age": 21},), "age=21, name=Ana"), (({},), "")], runner=lambda fn, a: fn(**a[0]))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over sorted keys and build the text with +.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`**info` collects keyword arguments into a dict; join a generator over its sorted items.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over sorted keys and build the text with +.

```python
def describe(**info):
    text = ""
    for key in sorted(info):
        if text:
            text += ", "
        text += key + "=" + str(info[key])
    return text
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `**info` collects keyword arguments into a dict; join a generator over its sorted items.

```python
def describe(**info):
    return ", ".join(f"{k}={v}" for k, v in sorted(info.items()))
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over sorted keys and build the text with +. That costs O(k log k) time and O(k) space."

"I can do better: `**info` collects keyword arguments into a dict; join a generator over its sorted items. That's O(k log k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/keyword-arguments-to-text)

<a id="q12"></a>
### Q12. n-th Fibonacci Number

🟡 Interview standard · *Memoisation*

Return the n-th Fibonacci number (fib(0) = 0, fib(1) = 1).

**Examples**

- `n = 10` → `55`
- `n = 0` → `0`
- `n = 20` → `6765`

In [ ]:
def fib(n):
    """
    Return the n-th Fibonacci number (fib(0) = 0, fib(1) = 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fib, [
    ((10,), 55),
    ((0,), 0),
    ((20,), 6765),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Plain recursion fib(n − 1) + fib(n − 2) — recomputes the same values again and again.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Cache results with `functools.lru_cache` so each fib(k) is computed once.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Plain recursion fib(n − 1) + fib(n − 2) — recomputes the same values again and again.

```python
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Cache results with `functools.lru_cache` so each fib(k) is computed once.

```python
def fib(n):
    from functools import lru_cache

    @lru_cache(maxsize=None)
    def f(k):
        return k if k < 2 else f(k - 1) + f(k - 2)
    return f(n)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Plain recursion fib(n − 1) + fib(n − 2) — recomputes the same values again and again. That costs O(2ⁿ) time and O(n) space."

"I can do better: Cache results with `functools.lru_cache` so each fib(k) is computed once. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 5`

1. f() returns 1
2. f() returns 0
3. f() returns 1
4. f() returns 2
5. f() returns 3
6. f() returns 5
7. fib() returns 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/n-th-fibonacci-number)

<a id="q13"></a>
### Q13. Fast Power

🟡 Interview standard · *Divide and conquer*

Return base \*\* exp for a non-negative integer exp using recursion.

**Examples**

- `base = 2, exp = 10` → `1024`
- `base = 3, exp = 0` → `1`
- `base = 5, exp = 3` → `125`

In [ ]:
def power(base, exp):
    """
    Return base ** exp for a non-negative integer exp using recursion.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(power, [
    ((2, 10), 1024),
    ((3, 0), 1),
    ((5, 3), 125),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

power(b, e) = b × power(b, e − 1) — e recursive calls.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Halve the exponent: power(b, e) = power(b, e // 2)², times b when e is odd.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** power(b, e) = b × power(b, e − 1) — e recursive calls.

```python
def power(base, exp):
    if exp == 0:
        return 1
    return base * power(base, exp - 1)
```

**Cost:** O(e) time, O(e) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Halve the exponent: power(b, e) = power(b, e // 2)², times b when e is odd.

```python
def power(base, exp):
    if exp == 0:
        return 1
    half = power(base, exp // 2)
    return half * half if exp % 2 == 0 else half * half * base
```

**Cost:** O(log e) time, O(log e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: power(b, e) = b × power(b, e − 1) — e recursive calls. That costs O(e) time and O(e) space."

"I can do better: Halve the exponent: power(b, e) = power(b, e // 2)², times b when e is odd. That's O(log e) time and O(log e) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `base = 2, exp = 10`

1. power() returns 1
2. power() returns 2
3. power() returns 4
4. power() returns 32
5. power() returns 1024

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/fast-power)

<a id="q14"></a>
### Q14. Apply an Operation by Name

🟡 Interview standard · *Functions as values*

Apply `op` ('double', 'square' or 'negate') to every number and return the new list.

**Examples**

- `nums = [1, 2, 3], op = 'double'` → `[2, 4, 6]`
- `nums = [1, 2, 3], op = 'square'` → `[1, 4, 9]`
- `nums = [4], op = 'negate'` → `[-4]`

In [ ]:
def apply_op(nums, op):
    """
    Apply op ('double', 'square' or 'negate') to every number and return the new
    list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(apply_op, [
    (([1, 2, 3], 'double'), [2, 4, 6]),
    (([1, 2, 3], 'square'), [1, 4, 9]),
    (([4], 'negate'), [-4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif inside the loop for every number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store the functions in a dict and `map` the chosen one.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An if/elif inside the loop for every number.

```python
def apply_op(nums, op):
    result = []
    for x in nums:
        if op == "double":
            result.append(x * 2)
        elif op == "square":
            result.append(x * x)
        elif op == "negate":
            result.append(-x)
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store the functions in a dict and `map` the chosen one.

```python
def apply_op(nums, op):
    ops = {"double": lambda x: x * 2, "square": lambda x: x * x, "negate": lambda x: -x}
    return list(map(ops[op], nums))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif inside the loop for every number. That costs O(n) time and O(n) space."

"I can do better: Store the functions in a dict and `map` the chosen one. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/apply-an-operation-by-name)

<a id="q15"></a>
### Q15. Scale With a Closure

🟡 Interview standard · *Closures*

Return every number multiplied by k, using a helper function built by `make_scaler(k)` in the optimal answer.

**Examples**

- `nums = [1, 2, 3], k = 10` → `[10, 20, 30]`
- `nums = [], k = 5` → `[]`

In [ ]:
def scale_all(nums, k):
    """
    Return every number multiplied by k, using a helper function built by
    make_scaler(k) in the optimal answer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scale_all, [
    (([1, 2, 3], 10), [10, 20, 30]),
    (([], 5), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A plain loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An inner function remembers k from the outer function (a closure).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A plain loop.

```python
def scale_all(nums, k):
    out = []
    for x in nums:
        out.append(x * k)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An inner function remembers k from the outer function (a closure).

```python
def scale_all(nums, k):
    def make_scaler(factor):
        def scale(x):
            return x * factor
        return scale

    scale = make_scaler(k)
    return [scale(x) for x in nums]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A plain loop. That costs O(n) time and O(n) space."

"I can do better: An inner function remembers k from the outer function (a closure). That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3], k = 10`

1. make\_scaler() returns &lt;function scale\_all.&lt;locals&gt;.make\_sca...
2. scale() returns 10
3. scale() returns 20
4. scale() returns 30
5. scale\_all() returns [10, 20, 30]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/scale-with-a-closure)

<a id="q16"></a>
### Q16. Validate an Order

🟡 Interview standard · *Guard clauses*

Return 'ok', or the first problem: 'no items', 'bad quantity' (qty ≤ 0) or 'too expensive' (total &gt; budget). Order is a list of (price, qty).

**Examples**

- `order = [], budget = 100` → `'no items'`
- `order = [(10, 0)], budget = 100` → `'bad quantity'`
- `order = [(60, 2)], budget = 100` → `'too expensive'`

In [ ]:
def validate(order, budget):
    """
    Return 'ok', or the first problem: 'no items', 'bad quantity' (qty ≤ 0) or
    'too expensive' (total > budget). Order is a list of (price, qty).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(validate, [
    (([], 100), 'no items'),
    (([(10, 0)], 100), 'bad quantity'),
    (([(60, 2)], 100), 'too expensive'),
    (([(20, 2)], 100), 'ok'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested ifs, with the happy path deep inside.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Guard clauses: return early on each problem, then the happy path is flat.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Nested ifs, with the happy path deep inside.

```python
def validate(order, budget):
    if order:
        valid_qty = True
        for price, qty in order:
            if qty <= 0:
                valid_qty = False
        if valid_qty:
            if sum(p * q for p, q in order) <= budget:
                return "ok"
            else:
                return "too expensive"
        else:
            return "bad quantity"
    else:
        return "no items"
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Guard clauses: return early on each problem, then the happy path is flat.

```python
def validate(order, budget):
    if not order:
        return "no items"
    if any(qty <= 0 for _, qty in order):
        return "bad quantity"
    if sum(price * qty for price, qty in order) > budget:
        return "too expensive"
    return "ok"
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested ifs, with the happy path deep inside. That costs O(n) time and O(1) space."

"I can do better: Guard clauses: return early on each problem, then the happy path is flat. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/validate-an-order)

<a id="q17"></a>
### Q17. Binary String, Recursively

🟡 Interview standard · *Recursion on n // 2*

Return the binary representation of a non-negative integer as a string.

**Examples**

- `n = 5` → `'101'`
- `n = 0` → `'0'`
- `n = 8` → `'1000'`

In [ ]:
def to_binary(n):
    """
    Return the binary representation of a non-negative integer as a string.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_binary, [
    ((5,), '101'),
    ((0,), '0'),
    ((8,), '1000'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

binary(n) = binary(n // 2) + last bit; base case n &lt; 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`bin(n)[2:]` or `format(n, 'b')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** binary(n) = binary(n // 2) + last bit; base case n &lt; 2.

```python
def to_binary(n):
    if n < 2:
        return str(n)
    return to_binary(n // 2) + str(n % 2)
```

**Cost:** O(log n) time, O(log n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `bin(n)[2:]` or `format(n, 'b')`.

```python
def to_binary(n):
    return format(n, "b")
```

**Cost:** O(log n) time, O(log n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: binary(n) = binary(n // 2) + last bit; base case n &lt; 2. That costs O(log n) time and O(log n) space."

"I can do better: `bin(n)[2:]` or `format(n, 'b')`. That's O(log n) time and O(log n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/binary-string-recursively)

<a id="q18"></a>
### Q18. Tower of Hanoi Moves

🟡 Interview standard · *Recursion vs formula*

How many moves are needed to move n discs in the Tower of Hanoi puzzle?

**Examples**

- `n = 3` → `7`
- `n = 1` → `1`
- `n = 10` → `1023`

In [ ]:
def hanoi_moves(n):
    """
    How many moves are needed to move n discs in the Tower of Hanoi puzzle?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hanoi_moves, [
    ((3,), 7),
    ((1,), 1),
    ((10,), 1023),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive: move n − 1 discs away, move the big disc, move n − 1 back: T(n) = 2T(n − 1) + 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Solve the recurrence: T(n) = 2ⁿ − 1.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursive: move n − 1 discs away, move the big disc, move n − 1 back: T(n) = 2T(n − 1) + 1.

```python
def hanoi_moves(n):
    if n == 0:
        return 0
    return hanoi_moves(n - 1) + 1 + hanoi_moves(n - 1)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Solve the recurrence: T(n) = 2ⁿ − 1.

```python
def hanoi_moves(n):
    return 2 ** n - 1
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive: move n − 1 discs away, move the big disc, move n − 1 back: T(n) = 2T(n − 1) + 1. That costs O(2ⁿ) time and O(n) space."

"I can do better: Solve the recurrence: T(n) = 2ⁿ − 1. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/tower-of-hanoi-moves)

<a id="q19"></a>
### Q19. Climbing Stairs

🟡 Interview standard · *Recursion + memo*

You climb 1 or 2 steps at a time. How many ways to reach step n?

**Examples**

- `n = 3` → `3`
- `n = 5` → `8`
- `n = 1` → `1`

In [ ]:
def stairs(n):
    """
    You climb 1 or 2 steps at a time. How many ways to reach step n?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stairs, [
    ((3,), 3),
    ((5,), 8),
    ((1,), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

ways(n) = ways(n − 1) + ways(n − 2) with plain recursion.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same recursion with a memo dict, so each step is solved once.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** ways(n) = ways(n − 1) + ways(n − 2) with plain recursion.

```python
def stairs(n):
    if n <= 2:
        return n
    return stairs(n - 1) + stairs(n - 2)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Same recursion with a memo dict, so each step is solved once.

```python
def stairs(n):
    memo = {1: 1, 2: 2}

    def ways(k):
        if k not in memo:
            memo[k] = ways(k - 1) + ways(k - 2)
        return memo[k]

    return ways(n)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: ways(n) = ways(n − 1) + ways(n − 2) with plain recursion. That costs O(2ⁿ) time and O(n) space."

"I can do better: Same recursion with a memo dict, so each step is solved once. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 5`

1. ways() returns 2
2. ways() returns 1
3. ways() returns 3
4. ways() returns 2
5. ways() returns 5
6. ways() returns 3
7. ways() returns 8
8. stairs() returns 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/climbing-stairs)

<a id="q20"></a>
### Q20. Flatten Any Depth

🔴 Stretch · *Recursion on nested lists*

Flatten lists nested to any depth: [1, [2, [3, [4]]], 5] → [1, 2, 3, 4, 5].

**Examples**

- `items = [1, [2, [3, [4]]], 5]` → `[1, 2, 3, 4, 5]`
- `items = []` → `[]`
- `items = [[[]]]` → `[]`

In [ ]:
def flatten(items):
    """
    Flatten lists nested to any depth: [1, [2, [3, [4]]], 5] → [1, 2, 3, 4, 5].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [
    (([1, [2, [3, [4]]], 5],), [1, 2, 3, 4, 5]),
    (([],), []),
    (([[[]]],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively flatten each item and concatenate the results (lots of list copying).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Recurse with one shared output list, appending leaves as you find them.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Recursively flatten each item and concatenate the results (lots of list copying).

```python
def flatten(items):
    result = []
    for x in items:
        if isinstance(x, list):
            result = result + flatten(x)
        else:
            result = result + [x]
    return result
```

**Cost:** O(n · depth) time, O(n · depth) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Recurse with one shared output list, appending leaves as you find them.

```python
def flatten(items):
    out = []

    def walk(xs):
        for x in xs:
            if isinstance(x, list):
                walk(x)
            else:
                out.append(x)

    walk(items)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively flatten each item and concatenate the results (lots of list copying). That costs O(n · depth) time and O(n · depth) space."

"I can do better: Recurse with one shared output list, appending leaves as you find them. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `items = [1, [2, [3]], 4]`

1. walk() returns None
2. walk() returns None
3. walk() returns None
4. flatten() returns [1, 2, 3, 4]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/functions/flatten-any-depth)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Parameter vs argument?</b></summary>

A parameter is the variable name in the function definition; an argument is the actual value passed when calling it.

</details>

<details>
<summary><b>2. What does a function return if there is no return statement?</b></summary>

`None`.

</details>

<details>
<summary><b>3. What are \*args and \*\*kwargs?</b></summary>

`*args` collects extra positional arguments into a tuple; `**kwargs` collects extra keyword arguments into a dict. The names are conventions; the stars matter.

</details>

<details>
<summary><b>4. Why is def f(x, items=[]) dangerous?</b></summary>

Default values are created once, when the function is defined, so every call shares the same list and changes accumulate. Use `items=None` and create a new list inside.

</details>

<details>
<summary><b>5. Is Python pass-by-value or pass-by-reference?</b></summary>

Neither exactly — it's 'pass by object reference'. The function receives a reference to the same object: mutating a list affects the caller, but rebinding the parameter name does not.

</details>

<details>
<summary><b>6. What is a lambda and when should you use it?</b></summary>

An anonymous one-expression function, e.g. `lambda x: x * 2`. Use it for short throwaway functions such as sort keys; use `def` for anything longer or reused.

</details>

<details>
<summary><b>7. What is a closure?</b></summary>

A nested function that captures variables from its enclosing function's scope, keeping them alive after the outer function returns — e.g. a multiplier factory.

</details>

<details>
<summary><b>8. What is the LEGB rule?</b></summary>

The order Python searches for a name: Local, Enclosing, Global, Built-in.

</details>

<details>
<summary><b>9. Recursion vs iteration?</b></summary>

Recursion expresses self-similar problems elegantly but uses a stack frame per call and is limited in depth. Iteration uses constant stack space and is usually faster. Both can solve the same problems.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why functions?
- [ ] I can explain: Parameters, arguments and return
- [ ] I can explain: Flexible arguments: \*args and \*\*kwargs
- [ ] I can explain: Scope: where variables live
- [ ] I can explain: Functions are values: lambda, map, closures
- [ ] I can explain: Recursion
- [ ] I can explain: Making recursion fast: memoisation

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Validate an Order
- [ ] [Q17](#q17) Binary String, Recursively
- [ ] [Q18](#q18) Tower of Hanoi Moves
- [ ] [Q19](#q19) Climbing Stairs
- [ ] [Q20](#q20) Flatten Any Depth

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.